# 01 - Normalize Curated Corpus v3

Normalizes `legal_documents_curated.csv`, keeps official law/article records as the main RAG corpus, separates QA rows as auxiliary data, and writes a normalization report. This notebook does not build indexes and does not run fine-tuning.

In [ ]:
from pathlib import Path
import json
import sys

try:
    from google.colab import drive
    drive.mount('/content/drive')
except ModuleNotFoundError:
    print('Not running in Google Colab; using local filesystem paths.')

DRIVE_ROOT = Path('/content/drive/MyDrive/rag')
sys.path.insert(0, str(DRIVE_ROOT))

config_path = DRIVE_ROOT / 'project_config.json'
if not config_path.exists():
    raise FileNotFoundError('Run 00_drive_setup_and_file_check.ipynb first to create project_config.json')

config = json.loads(config_path.read_text(encoding='utf-8'))
DRIVE_ROOT

In [ ]:
from src.normalize_corpus import OutputPaths, normalize_curated_corpus

input_csv = DRIVE_ROOT / config['raw_curated_csv']
if not input_csv.exists():
    raise FileNotFoundError(f'Missing input corpus: {input_csv}')

outputs = OutputPaths(
    main_csv=DRIVE_ROOT / config['main_law_corpus_csv'],
    main_jsonl=DRIVE_ROOT / config['main_law_corpus_jsonl'],
    qa_csv=DRIVE_ROOT / config['qa_auxiliary_csv'],
    rejected_csv=DRIVE_ROOT / config['rejected_review_csv'],
    report_json=DRIVE_ROOT / config['normalization_report_json'],
)

report = normalize_curated_corpus(input_csv, outputs)
report

In [ ]:
import pandas as pd

main_df = pd.read_csv(outputs.main_csv)
qa_df = pd.read_csv(outputs.qa_csv)
rejected_df = pd.read_csv(outputs.rejected_csv)

print('Main official law corpus:', outputs.main_csv, main_df.shape)
print('Main JSONL:', outputs.main_jsonl)
print('QA auxiliary:', outputs.qa_csv, qa_df.shape)
print('Rejected/review:', outputs.rejected_csv, rejected_df.shape)
print('Report:', outputs.report_json)

main_df.head(3)

In [ ]:
required_columns = [
    'record_id',
    'doc_key',
    'article_key',
    'source_type',
    'authority_level',
    'official_source',
    'law_name_raw',
    'law_name_norm',
    'law_no_norm',
    'mevzuat_no',
    'mevzuat_tur',
    'mevzuat_tertip',
    'source_url',
    'article_no_raw',
    'article_no_norm',
    'article_type',
    'article_number',
    'article_body',
    'retrieval_text',
    'generation_text',
    'citation_label',
    'canonical_source',
    'category',
    'publication_date',
    'text_length',
    'body_length',
    'quality_flag',
    'duplicate_group_id',
]

missing_columns = [col for col in required_columns if col not in main_df.columns]
if missing_columns:
    raise AssertionError(f'Missing normalized columns: {missing_columns}')

print('Schema check passed.')
print(main_df['quality_flag'].value_counts(dropna=False).to_string())

In [ ]:
preview_columns = ['doc_key', 'article_key', 'law_name_norm', 'article_no_norm', 'citation_label', 'body_length']
main_df[preview_columns].sample(min(5, len(main_df)), random_state=42) if len(main_df) else main_df[preview_columns]

Next step after reviewing the report: rebuild the dense/BM25 indexes from `retrieval_text`, then rerun retrieval evaluation against the locked benchmark.